In [1]:
"""Library imports"""
import pandas as pd
import numpy as np
import datetime
import calendar
import glob
import os
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [2]:
"""Basic Options"""
# Paste the path to the folder with your data between the quotation marks below.
PATH_NEWARE_DATA = r"C:\Jiggler Data\Current data\Cycling Data\EJB_Jiggler1_487579_42V_20C_230216_NCRHA\SaturdayExport\Combined"
PATH_JIGGLER_DATA = r"C:\Jiggler Data\Current data\Cycling Data\EJB_Jiggler1_487579_42V_20C_230216_NCRHA\SaturdayExport\Combined"

# Type the desired name of the output folder. If you put nothing, it will 
# default to a subfolder in the input folder 
OUTPUT_PATH = r"C:\Jiggler Data\Current data\Cycling Data\EJB_Jiggler1_487579_42V_20C_230216_NCRHA\SaturdayExport\Combined"

In [3]:
# Grab Filenames
fname_neware = sorted(glob.glob(PATH_NEWARE_DATA + '\*.xls'))
fname_jiggler = sorted(glob.glob(PATH_JIGGLER_DATA + '\*.csv'))

# Import a single file
df  = pd.read_excel(fname_neware[0], sheet_name = "sheet1", skiprows = 2, header = 0,
                     usecols=[2, 4, 7, 11])

jiggler_df_raw = pd.read_csv(fname_jiggler[0])
jiggler_df_raw = jiggler_df_raw.drop(jiggler_df_raw.columns[0], axis=1)
# Dropping undesirable row after header row
df = df.drop(0)

In [4]:
"""Neware Parsing

General Idea, parse the data using the Record ID rows with integer values.

We want the row data corresponding to the integer index of the Record ID column 
in the raw neware data files. We will broadcast the cycle state 
(I.E Charge/Discharge/CVCharge/....) and the integer step number will be saved 
as the index of the parsed data
"""

'Neware Parsing\n\nTo parse the neware data, we want the row data corresponding to the integer index\nof the Record ID column in the raw neware data files. We will broadcast the \ncycle state (I.E Charge/Discharge/CVCharge/....) and the integer step number\nwill be saved as the index of the parsed data'

In [5]:
# Slicing the Record ID object type column, by object type.
record_id_ints = df["Record ID"].map(type) == int
record_id_strings = df["Record ID"].loc[df["Record ID"].str.contains("^") == True]
# Indexes all numerical entries
# record_id_nums = df["Record ID"].loc[df["Record ID"].astype(str).str.isnumeric() == True]

In [6]:
"""Constructing new Record ID column with the cycle state 
   (I.E Charge/Discharge/CVCharge etc...) state as each entry"""

index_list = record_id_strings.index.values
# Add the index for the last row +1 to account for the deleted non-int row
index_list = np.append(index_list, df["Record ID"].index.values[-1]+1)
string_list = record_id_strings.values

# Filling in blanks with correct step
cycle_state = []
for i in range(len(index_list)):
    if i == 0:
        continue
    steps_of_state = index_list[i] - index_list[i-1]
    cycle_state = cycle_state + (steps_of_state*[string_list[i-1]])

# Converting List to a Series
cycle_state_df = pd.DataFrame({"Charge/Discharge Step" : cycle_state})
# Adjust index to match
cycle_state_df.index += 1

In [24]:
"""Slicing out our data"""

"""Float Eliminator
Due to how Pandas handles multiple data types in a single column, if a float ends
with all zeroes (I.E 3121.0000) it will try and save memory by storing the value
as an integer. This means that on rare occasion the rows with CYCLE ID values
will appear like integers.

This is only occasionally an issue, but to be safe we use the integers to slice
the datetime data, then filter out any remaining integer or float values in the
column. We then use the resultant index to slice the remaining columns"""

DT_raw = df["Realtime"].loc[df["Record ID"].map(type) == int]
DT_no_int = DT_raw.loc[DT_raw.map(type) != int]
DT_no_int_no_float = DT_no_int[DT_no_int.map(type) != float]

# Using the filetered datetime column as the new slicing index
Datetime_col = DT_no_int_no_float
slice_index = DT_no_int_no_float.index.values

cycle_state_col = cycle_state_df["Charge/Discharge Step"].loc[slice_index]
try:
    Voltage_col = df["Voltage(V)"].loc[slice_index]
except:
    Voltage_col = df["Voltage(mV)"].loc[slice_index]

# Construct a new dataframe with the desired values
Neware_df = pd.concat([cycle_state_col, Voltage_col, Datetime_col], axis = 1)
Neware_df.rename(columns = {"Realtime" : "Datetime"}, inplace = True)

In [11]:
"""Determining Relevant Combined Experiment time"""
jiggler_df_raw["Datetime"] = pd.to_datetime(jiggler_df_raw["Datetime"], format = '%Y-%m-%d %H:%M:%S')
Neware_df["Datetime"] = pd.to_datetime(Neware_df["Datetime"], format = '%Y-%m-%d %H:%M:%S')

# Start and end times of the neware file
start = Neware_df["Datetime"].iloc[0]
end = Neware_df["Datetime"].iloc[-1]

# Slicing from 20 minutes before the neware started until 20 minutes after the neware finished.
jiggler_df_i = jiggler_df_raw.loc[(jiggler_df_raw["Datetime"] > (start - pd.Timedelta(+20, 'm')))]
jiggler_df = jiggler_df_i.loc[(jiggler_df_raw["Datetime"] < (end - pd.Timedelta(-20, 'm')))]

# Adding Column for time in seconds since the "Start of experiment"
seconds_jig = (jiggler_df["Datetime"] - Neware_df["Datetime"].iloc[0]).astype("timedelta64[s]")
seconds_new = (Neware_df["Datetime"] - Neware_df["Datetime"].iloc[0]).astype("timedelta64[s]")
jiggler_df["TSSoE in hours"] = seconds_jig/3600
Neware_df["TSSoE in hours"] = seconds_new/3600

ValueError: mixed datetimes and integers in passed array

In [ ]:
# Exporting DataFrames
if os.path.exists(f"{OUTPUT_PATH}\\DataFrames") == False:
                os.mkdir(f"{OUTPUT_PATH}\\DataFrames")

start_fmt = start.strftime("%d %b %Y")
end_fmt = end.strftime("%d %b %Y")

Neware_df.to_csv(f"{OUTPUT_PATH}\\DataFrames\\Neware_df_{start_fmt}-{end_fmt}.csv")
jiggler_df.to_csv(f"{OUTPUT_PATH}\\DataFrames\\Jiggler_df_{start_fmt}-{end_fmt}.csv")

PermissionError: [Errno 13] Permission denied: 'C:\\Jiggler Data\\Current data\\Cycling Data\\EJB_Jiggler1_487579_42V_20C_230216_NCRHA\\SaturdayExport\\Combined\\DataFrames\\Neware_df_16 Feb 2023-20 Feb 2023.csv'